In [1]:
# ==============================================================================
# DATAVIEW: EXPLORAÇÃO E ANÁLISE DE DADOS DE VENDAS
# ==============================================================================
# Passo 1: Importação de bibliotecas
import pandas as pd
import numpy as np
import random
import os
import json
import re
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta



ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# ==============================================================================
# FUNÇÕES DE EXTRAÇÃO E INSPEÇÃO (RF01, RF02)
# ==============================================================================
def gerar_dataset_vendas(n_registros=150, seed=42):
    """RF01: Gera dataset sintético com dados propositalmente sujos."""
    random.seed(seed)
    np.random.seed(seed)

    produtos = ['Notebook', 'Smartphone', 'Tablet', 'Monitor', 'Teclado', 'Mouse']
    precos = {'Notebook': 3500, 'Smartphone': 2200, 'Tablet': 1800, 'Monitor': 1200, 'Teclado': 250, 'Mouse': 120}
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares", "Tablet": "Celulares", "Monitor": "Computadores", "Teclado": "Periféricos", "Mouse": "Periféricos"}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
    clientes = [f"Cliente_{i:03d}" for i in range(1, 31)]
    data_inicio = datetime(2024, 1, 1)

    dados = []
    for i in range(n_registros):
        produto = random.choice(produtos)
        quantidade = random.randint(1, 10)
        preco = precos[produto]
        data = data_inicio + timedelta(days=random.randint(0, 364))

        # Sujeira intencional
        if random.random() < 0.05: quantidade = None
        if random.random() < 0.04: preco = None
        if random.random() < 0.03: produto = "  " + produto + " "

        data_str = data.strftime("%Y-%m-%d") if random.random() > 0.02 else "DATA INVALIDA"

        dados.append({
            "id_venda": i + 1, "data_venda": data_str, "cliente": random.choice(clientes),
            "produto": produto, "categoria": categorias.get(produto.strip(), "Outros"),
            "regiao": random.choice(regioes), "quantidade": quantidade, "preco_unitario": preco
        })

    df = pd.DataFrame(dados)
    os.makedirs('data/raw', exist_ok=True)
    df.to_csv("data/raw/vendas.csv", index=False)
    return df



In [ ]:
def inspecionar_dados(df):
    """RF02: Exibe o shape e os valores nulos iniciais."""
    print("\n=== INSPEÇÃO INICIAL ===")
    print(f"Shape: {df.shape} | Valores Nulos:\n{df.isnull().sum()}")



In [ ]:
# ==============================================================================
# FUNÇÕES DE LIMPEZA E TRANSFORMAÇÃO (RF03, RF04, RF05)
# ==============================================================================
def limpar_dados(df):
    """RF03: Limpa regex, datas inválidas e nulos cruciais."""
    df = df.copy()
    colunas_texto = df.select_dtypes(include="object").columns

    # Limpeza Regex
    for col in colunas_texto:
        df[col] = df[col].apply(lambda s: re.sub(r"\s+", " ", str(s)).strip() if pd.notna(s) else s)

    # Conversão de Datas e remoção de nulos
    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")
    df = df.dropna(subset=["data_venda", "quantidade", "preco_unitario"])

    # Correção de Tipos
    df["quantidade"] = df["quantidade"].astype(int)
    df["preco_unitario"] = df["preco_unitario"].astype(float)

    os.makedirs("data/processed/v1_com_outliers", exist_ok=True)
    df.to_csv("data/processed/v1_com_outliers/vendas_v1.csv", index=False)
    return df



In [ ]:
def tratar_outliers(df, colunas, fator=1.5):
    """RF04: Remove outliers numéricos usando o método IQR."""
    df = df.copy()
    print("\n=== TRATAMENTO DE OUTLIERS ===")
    for col in colunas:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lim_inf, lim_sup = q1 - fator * iqr, q3 + fator * iqr

        n_out = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()
        print(f"{col}: {n_out} outliers removidos.")
        df = df[(df[col] >= lim_inf) & (df[col] <= lim_sup)]

    os.makedirs("data/processed/v2_outliers_tratado", exist_ok=True)
    df.to_csv("data/processed/v2_outliers_tratado/vendas_v2.csv", index=False)
    return df



In [ ]:
def criar_colunas_derivadas(df):
    """RF05: Deriva receita_total, mês, trimestre e segmentação de valor de item."""
    df = df.copy()
    df["receita_total"] = df["quantidade"] * df["preco_unitario"]
    df["mes"] = df["data_venda"].dt.month
    df["trimestre"] = df["data_venda"].dt.quarter.apply(lambda q: f"Q{q}")
    df["ano"] = df["data_venda"].dt.year

    condicoes = [df["receita_total"] < 500, (df["receita_total"] >= 500) & (df["receita_total"] < 5000), df["receita_total"] >= 5000]
    df["faixa_receita_item"] = np.select(condicoes, ["Baixo Valor", "Médio Valor", "Alto Valor"], default="N/D")
    return df



In [ ]:
# ==============================================================================
# FUNÇÕES DE ANÁLISE E INTELIGÊNCIA (RF06, RF07, RF08)
# ==============================================================================
def calcular_metricas(df):
    """RF06: Agrupa dados por mês, produto, categoria e região."""
    return {
        "por_mes": df.groupby("mes").agg(receita_total=("receita_total", "sum"), quantidade=("quantidade", "sum")).reset_index().sort_values("mes"),
        "top_produtos": df.groupby("produto")["receita_total"].sum().sort_values(ascending=False).head(5).reset_index(),
        "por_categoria": df.groupby("categoria")["receita_total"].sum().reset_index().sort_values("receita_total", ascending=False),
        "por_regiao": df.groupby("regiao").agg(receita_total=("receita_total", "sum"), media_ticket=("receita_total", "mean")).reset_index()
    }



In [ ]:
def segmentar_clientes(df):
    """RF07: Segmenta clientes em níveis (Bronze, Prata, Ouro) via Lambda."""
    clientes_df = df.groupby("cliente")["receita_total"].sum().reset_index()
    clientes_df.columns = ["cliente", "total_gasto"]
    clientes_df["segmento"] = clientes_df["total_gasto"].apply(lambda g: "Ouro" if g > 15000 else ("Prata" if g >= 5000 else "Bronze"))
    return clientes_df.sort_values("total_gasto", ascending=False)



In [ ]:
def calcular_estatisticas_numpy(df):
    """RF08: Calcula estatísticas diretas sobre Arrays NumPy."""
    receitas = df["receita_total"].to_numpy()
    media = float(np.mean(receitas))
    return {
        "media": media, "mediana": float(np.median(receitas)), "desvio_padrao": float(np.std(receitas)),
        "total": float(np.sum(receitas)), "p25": float(np.percentile(receitas, 25)), "p75": float(np.percentile(receitas, 75)),
        "acima_da_media": int((receitas > media).sum())
    }



In [ ]:
# ==============================================================================
# FUNÇÕES DE VISUALIZAÇÃO E RELATÓRIO (RF09 + Diferenciais)
# ==============================================================================
def gerar_visualizacoes(df, metricas, output_dir="outputs/graficos"):
    """RF09 e Diferencial: Gera os gráficos básicos e o Pairplot."""
    os.makedirs(output_dir, exist_ok=True)
    sns.set_theme(style="whitegrid", palette="muted")

    # 1. Linha (Receita por Mês)
    plt.figure(figsize=(10, 4))
    plt.plot(metricas["por_mes"]["mes"], metricas["por_mes"]["receita_total"], marker="o", color="teal")
    plt.title("Receita Total por Mês")
    plt.savefig(f"{output_dir}/receita_por_mes.png", dpi=120, bbox_inches='tight')
    plt.close()

    # 2. Barras (Top Produtos)
    plt.figure(figsize=(10, 4))
    sns.barplot(data=metricas["top_produtos"], y="produto", x="receita_total")
    plt.title("Top 5 Produtos")
    plt.savefig(f"{output_dir}/top_produtos.png", dpi=120, bbox_inches='tight')
    plt.close()

    # 3. Boxplot (Região)
    plt.figure(figsize=(10, 4))
    sns.boxplot(data=df, x="regiao", y="receita_total")
    plt.title("Dispersão por Região")
    plt.savefig(f"{output_dir}/dist_regiao.png", dpi=120, bbox_inches='tight')
    plt.close()

    # 4. PAIRPLOT (DIFERENCIAL)
    print("\n[DIFERENCIAL] Gerando Pairplot Multivariado...")
    colunas_interesse = ['quantidade', 'preco_unitario', 'receita_total', 'regiao']
    g = sns.pairplot(df[colunas_interesse], hue="regiao", palette="husl", corner=True, diag_kind="kde")
    g.fig.suptitle("Análise Multivariada: Preço, Quantidade e Receita por Região", y=1.05)
    g.savefig(f"{output_dir}/pairplot_multivariado.png", dpi=150, bbox_inches='tight')
    plt.close()

def gerar_relatorio_executivo(metricas, stats_numpy, clientes_df):
    """DIFERENCIAL: Gera insights textuais baseados nos dados."""
    print("\n" + "="*60)
    print("      RELATÓRIO EXECUTIVO TRIMESTRAL DE VENDAS")
    print("="*60)

    melhor_mes = metricas["por_mes"].loc[metricas["por_mes"]['receita_total'].idxmax()]
    perc_ouro = (clientes_df['segmento'] == 'Ouro').mean() * 100
    top_produto = metricas["top_produtos"].iloc[0]

    print(f"1. SAZONALIDADE: O mês {int(melhor_mes['mes'])} liderou com R$ {melhor_mes['receita_total']:,.2f}.")
    print(f"2. CLIENTES: Segmento 'Ouro' compõe apenas {perc_ouro:.1f}% da base. Criar programa de fidelidade.")
    print(f"3. PRODUTO LÍDER: '{top_produto['produto']}' gerou R$ {top_produto['receita_total']:,.2f}.")
    print(f"4. TICKET MÉDIO: R$ {stats_numpy['media']:.2f}")
    print("="*60 + "\n")



In [ ]:
# ==============================================================================
# PIPELINE PRINCIPAL (RF10)
# ==============================================================================
def executar_pipeline_dataview():
    """RF10: Orquestra todas as funções do projeto de ponta a ponta."""
    print("Iniciando Pipeline DataView...")

    # 1. Dados Brutos
    df_bruto = gerar_dataset_vendas()
    inspecionar_dados(df_bruto)

    # 2. Limpeza e Tratamento
    df_v1 = limpar_dados(df_bruto)

    # Criação temporária de receita para limpar outliers baseados no valor final
    df_temp = df_v1.copy()
    df_temp["receita_total"] = df_temp["quantidade"] * df_temp["preco_unitario"]
    df_v2 = tratar_outliers(df_temp, ["quantidade", "receita_total"])
    df_v2 = df_v2.drop(columns=["receita_total"])

    # 3. Transformação
    df_final = criar_colunas_derivadas(df_v2)

    # 4. Agregações e Estatísticas
    metricas = calcular_metricas(df_final)
    clientes = segmentar_clientes(df_final)
    stats = calcular_estatisticas_numpy(df_final)

    # 5. Inteligência Visual e Textual
    gerar_visualizacoes(df_final, metricas)
    gerar_relatorio_executivo(metricas, stats, clientes)

    # 6. Exportação
    exportar_resultados(df_final, metricas, clientes, stats)

    print("Pipeline concluído com sucesso! Verifique as pastas 'data/' e 'outputs/'.")


In [ ]:
# ==============================================================================
# FUNÇÕES DE EXPORTAÇÃO (RF11, RF12)
# ==============================================================================
def exportar_resultados(df_final, metricas, clientes_df, stats_dict):
    """RF11 e RF12: Salva os CSVs e o JSON final."""
    os.makedirs("outputs", exist_ok=True)
    os.makedirs("data/final", exist_ok=True)

    metricas["por_mes"].to_csv("outputs/metricas_por_mes.csv", index=False)
    clientes_df.to_csv("outputs/segmentacao_clientes.csv", index=False)

    with open("outputs/estatisticas_gerais.json", "w", encoding="utf-8") as f:
        json.dump(stats_dict, f, indent=4, ensure_ascii=False)

    df_final.to_csv("data/final/vendas_final.csv", index=False)



In [ ]:

# ==============================================================================
# EXECUÇÃO DO PROJETO
# ==============================================================================
if __name__ == "__main__":
    executar_pipeline_dataview()

Iniciando Pipeline DataView...

=== INSPEÇÃO INICIAL ===
Shape: (150, 8) | Valores Nulos:
id_venda          0
data_venda        0
cliente           0
produto           0
categoria         0
regiao            0
quantidade        5
preco_unitario    2
dtype: int64

=== TRATAMENTO DE OUTLIERS ===
quantidade: 0 outliers removidos.
receita_total: 6 outliers removidos.

[DIFERENCIAL] Gerando Pairplot Multivariado...

      RELATÓRIO EXECUTIVO TRIMESTRAL DE VENDAS
1. SAZONALIDADE: O mês 4 liderou com R$ 143,040.00.
2. CLIENTES: Segmento 'Ouro' compõe apenas 86.2% da base. Criar programa de fidelidade.
3. PRODUTO LÍDER: 'Tablet' gerou R$ 261,000.00.
4. TICKET MÉDIO: R$ 6928.73

Pipeline concluído com sucesso! Verifique as pastas 'data/' e 'outputs/'.
